notebook_04_retrieval_docs.ipynb

Phase 4 — Retrieval Document Generation

Input:  product_entities_normalized_v2.json

         ProductCatalog_TRANSLATED_v2.xlsx  (for prose / Usage Instructions)

Output: retrieval_documents_structured_v2.json

        retrieval_documents_full_v2.json

In [1]:
import json
import pandas as pd

print('No API calls in this notebook — runs free.')

No API calls in this notebook — runs free.


In [2]:
# ── CELL 2 — Load normalized entities ────────────────────────────────────────
from google.colab import files

print('Upload product_entities_normalized_v2.json')
uploaded = files.upload()

with open('product_entities_normalized_v2.json', encoding='utf-8') as f:
    entities = json.load(f)

print(f'Loaded: {len(entities)} products')
entities_map = {e['product_id']: e for e in entities}

Upload product_entities_normalized_v2.json


Saving product_entities_normalized_v2.json to product_entities_normalized_v2.json
Loaded: 114 products


In [14]:
# Cell 3 addition — load Chinese prose too
df_cn = pd.read_excel('产品目录_CLEANED.xlsx')
cn_prose_map = {}
for _, row in df_cn.iterrows():
    pid = str(row['产品ID']).strip()
    cn_prose = str(row['中文使用说明']).strip()
    cn_prose_map[pid] = cn_prose if cn_prose != 'nan' else ''

# Updated build_full_document — Cell 6
def build_full_document(entity, prose, cn_prose=''):
    structured = build_structured_document(entity)

    sections = [structured]

    if prose and len(prose.strip()) > 20:
        sections.append(
            '━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━\n\n'
            'ORIGINAL PRODUCT DESCRIPTION\n' + prose.strip()
        )

    if cn_prose and len(cn_prose.strip()) > 20:
        sections.append(
            '━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━\n\n'
            '中文产品说明\n' + cn_prose.strip()
        )

    return '\n\n'.join(sections).strip()

# Cell 7 update — pass cn_prose to full doc builder
f_doc = build_full_document(entity, prose, cn_prose_map.get(pid, ''))

In [4]:
# ── CELL 4 — Helper: list to readable text ───────────────────────────────────
def list_to_text(values):
    """Convert a list to comma-separated string. Returns 'None' if empty."""
    if not values:
        return 'None'
    return ', '.join(str(v) for v in values)

In [5]:
# ── CELL 5 — Build STRUCTURED retrieval document ─────────────────────────────
# Structured document = clean English summary of all extracted entities.
# Both CN and EN product names included for bilingual retrieval.
# No prose / raw text — compact and precise.

def build_structured_document(entity):
    pid  = entity['product_id']
    return f"""PRODUCT SUMMARY

Product ID:           {pid}
Product Name:         {entity['product_name']}
Chinese Name:         {entity['product_name_cn']}
Product Types:        {list_to_text(entity.get('product_types', []))}

TARGET CROPS
{list_to_text(entity.get('target_crops', []))}

TARGET DISEASES
{list_to_text(entity.get('target_diseases', []))}

TARGET PESTS
{list_to_text(entity.get('target_pests', []))}

SYMPTOMS ADDRESSED
{list_to_text(entity.get('symptoms', []))}

ACTIVE INGREDIENTS
{list_to_text(entity.get('active_ingredients', []))}

DOSAGES
{list_to_text(entity.get('dosages', []))}

APPLICATION METHODS
{list_to_text(entity.get('application_methods', []))}

SAFETY WARNINGS
{list_to_text(entity.get('safety_warnings', []))}""".strip()



In [15]:
# ── CELL 6 — Build FULL retrieval document ───────────────────────────────────
# Full document = structured summary + original translated prose.
# The prose adds natural language context for semantic search —
# a farmer asking "how does this product work" will match the prose section.
# The structured section ensures precise field-level matches.

# ── CELL 6 updated — Build FULL retrieval document with Chinese prose ─────────
def build_full_document(entity, prose, cn_prose=''):
    structured = build_structured_document(entity)
    sections   = [structured]

    if prose and len(prose.strip()) > 20:
        sections.append(
            '━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━\n\n'
            'ORIGINAL PRODUCT DESCRIPTION\n' + prose.strip()
        )

    if cn_prose and len(cn_prose.strip()) > 20:
        sections.append(
            '━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━\n\n'
            '中文产品说明\n' + cn_prose.strip()
        )

    return '\n\n'.join(sections).strip()


In [16]:
# ── CELL 7 updated — Generate all documents ───────────────────────────────────
structured_docs = []
full_docs       = []

for entity in entities:
    pid      = entity['product_id']
    prose    = prose_map.get(pid, '')
    cn_prose = cn_prose_map.get(pid, '')

    s_doc = build_structured_document(entity)
    f_doc = build_full_document(entity, prose, cn_prose)

    structured_docs.append({
        'product_id':      pid,
        'product_name':    entity['product_name'],
        'product_name_cn': entity['product_name_cn'],
        'document':        s_doc,
    })

    full_docs.append({
        'product_id':      pid,
        'product_name':    entity['product_name'],
        'product_name_cn': entity['product_name_cn'],
        'document':        f_doc,
    })

print(f'Structured documents: {len(structured_docs)}')
print(f'Full documents:       {len(full_docs)}')

empty_s = sum(1 for d in structured_docs if not d['document'].strip())
empty_f = sum(1 for d in full_docs if not d['document'].strip())
print(f'Empty structured: {empty_s}')
print(f'Empty full:       {empty_f}')

# Verify CN prose attached
has_cn = sum(1 for d in full_docs if '中文产品说明' in d['document'])
print(f'Full docs with CN prose section: {has_cn} / {len(full_docs)}')

Structured documents: 114
Full documents:       114
Empty structured: 0
Empty full:       0
Full docs with CN prose section: 114 / 114


In [18]:
# ── CELL 8 — Spot check documents ────────────────────────────────────────────
# Check 3 products — microbial, pesticide, blank-ingredient fertilizer
for pid in ['AF0001', 'PN0001', 'AF0014']:
    s = next(d for d in structured_docs if d['product_id'] == pid)
    f = next(d for d in full_docs if d['product_id'] == 'AF0001')
print(f'Total length: {len(f["document"])} chars')
print(f'Has EN prose: {"ORIGINAL PRODUCT DESCRIPTION" in f["document"]}')
print(f'Has CN prose: {"中文产品说明" in f["document"]}')
print()
# Show the CN section
parts = f['document'].split('━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━')
print(f'Number of sections: {len(parts)}')
for i, part in enumerate(parts):
    print(f'\nSection {i+1} (first 150 chars):')
    print(part.strip()[:150])



Total length: 5105 chars
Has EN prose: True
Has CN prose: True

Number of sections: 3

Section 1 (first 150 chars):
PRODUCT SUMMARY
 
Product ID:           AF0001
Product Name:         Citrus Junliqing
Chinese Name:         柑橘菌立清
Product Types:        Growth Promote

Section 2 (first 150 chars):
ORIGINAL PRODUCT DESCRIPTION
Core control targets (diseases throughout the entire growth period of citrus)
1. Fungal diseases (core): root rot, downy 

Section 3 (first 150 chars):
中文产品说明
核心防治对象（柑橘全生育期病害）
1. 真菌病害（核心）：根腐病、霜霉病、灰霉病、紫斑病、炭疽病、叶斑病、枯萎病、猝倒病（苗期高发）；
2. 细菌病害（辅助）：软腐病、姜瘟病（通过菌群占位 + 抑菌代谢物抑制）；
3. 兼防病毒病（间接）：增强植株免疫力，减少病毒病引起的黄叶、干尖、花


In [19]:
# Quick prose check before saving
for pid in ['AF0001', 'PN0001']:
    f = next(d for d in full_docs if d['product_id'] == pid)
    has_separator = '━' in f['document']
    print(f"{pid}: has prose section = {has_separator}")
    print(f"  Full doc length: {len(f['document'])} chars")
    print(f"  Structured length: {len(next(d for d in structured_docs if d['product_id'] == pid)['document'])} chars")
    print()

AF0001: has prose section = True
  Full doc length: 5105 chars
  Structured length: 1111 chars

PN0001: has prose section = True
  Full doc length: 3499 chars
  Structured length: 1172 chars



In [20]:
# ── CELL 9 — Document length statistics ──────────────────────────────────────
s_lengths = [len(d['document']) for d in structured_docs]
f_lengths = [len(d['document']) for d in full_docs]

print('=== STRUCTURED DOCUMENT LENGTHS ===')
print(f'  Min:     {min(s_lengths):>6} chars')
print(f'  Max:     {max(s_lengths):>6} chars')
print(f'  Average: {sum(s_lengths)//len(s_lengths):>6} chars')

print('\n=== FULL DOCUMENT LENGTHS ===')
print(f'  Min:     {min(f_lengths):>6} chars')
print(f'  Max:     {max(f_lengths):>6} chars')
print(f'  Average: {sum(f_lengths)//len(f_lengths):>6} chars')

# Flag very short documents — may indicate missing data
short_s = [(d['product_id'], len(d['document'])) for d in structured_docs
           if len(d['document']) < 200]
if short_s:
    print(f'\nWARN: Short structured docs: {short_s}')
else:
    print('\nNo unusually short documents ✓')

=== STRUCTURED DOCUMENT LENGTHS ===
  Min:        419 chars
  Max:       2695 chars
  Average:   1184 chars

=== FULL DOCUMENT LENGTHS ===
  Min:        778 chars
  Max:       8941 chars
  Average:   4632 chars

No unusually short documents ✓


In [21]:
# ── CELL 10 — Bilingual search verification ──────────────────────────────────
# Confirm both CN and EN names appear in every document
print('=== BILINGUAL NAME CHECK ===')
missing_cn_in_doc = []
missing_en_in_doc = []

for d in structured_docs:
    entity = entities_map[d['product_id']]
    cn     = entity['product_name_cn']
    en     = entity['product_name']
    if cn and cn not in d['document']:
        missing_cn_in_doc.append(d['product_id'])
    if en and en not in d['document']:
        missing_en_in_doc.append(d['product_id'])

print(f'  Docs missing CN name: {missing_cn_in_doc if missing_cn_in_doc else "None ✓"}')
print(f'  Docs missing EN name: {missing_en_in_doc if missing_en_in_doc else "None ✓"}')


=== BILINGUAL NAME CHECK ===
  Docs missing CN name: None ✓
  Docs missing EN name: None ✓


In [22]:
# ── CELL 11 — Save ───────────────────────────────────────────────────────────
# Save structured
with open('retrieval_documents_structured_v2.json', 'w', encoding='utf-8') as f:
    json.dump(structured_docs, f, ensure_ascii=False, indent=2)
print(f'Saved → retrieval_documents_structured_v2.json  ({len(structured_docs)} docs)')

# Save full
with open('retrieval_documents_full_v2.json', 'w', encoding='utf-8') as f:
    json.dump(full_docs, f, ensure_ascii=False, indent=2)
print(f'Saved → retrieval_documents_full_v2.json  ({len(full_docs)} docs)')

# Verify
with open('retrieval_documents_structured_v2.json', encoding='utf-8') as f:
    verify_s = json.load(f)
with open('retrieval_documents_full_v2.json', encoding='utf-8') as f:
    verify_f = json.load(f)

print(f'\nVerified: {len(verify_s)} structured | {len(verify_f)} full ✓')
print(f'Keys: {list(verify_s[0].keys())}')

Saved → retrieval_documents_structured_v2.json  (114 docs)
Saved → retrieval_documents_full_v2.json  (114 docs)

Verified: 114 structured | 114 full ✓
Keys: ['product_id', 'product_name', 'product_name_cn', 'document']


In [23]:
# ── CELL 12 — Download ───────────────────────────────────────────────────────
from google.colab import files
files.download('retrieval_documents_structured_v2.json')
files.download('retrieval_documents_full_v2.json')

print("""
=== PHASE 4 COMPLETE ✓ ===
Input:  product_entities_normalized_v2.json
        ProductCatalog_TRANSLATED_v2.xlsx
Output: retrieval_documents_structured_v2.json
        retrieval_documents_full_v2.json
No API calls — 100% free
Next: AI-as-Judge → then notebook_05_embedding.ipynb
""")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


=== PHASE 4 COMPLETE ✓ ===
Input:  product_entities_normalized_v2.json
        ProductCatalog_TRANSLATED_v2.xlsx
Output: retrieval_documents_structured_v2.json
        retrieval_documents_full_v2.json
No API calls — 100% free
Next: AI-as-Judge → then notebook_05_embedding.ipynb



In [24]:
# ── CSV export — single line per document ────────────────────────────────────
import pandas as pd

# Collapse multiline document to single line for CSV
def to_single_line(text):
    return ' | '.join(line.strip() for line in text.splitlines() if line.strip())

# Structured CSV
pd.DataFrame([{
    'product_id':      d['product_id'],
    'product_name':    d['product_name'],
    'product_name_cn': d['product_name_cn'],
    'document':        to_single_line(d['document'])
} for d in structured_docs]).to_csv(
    'retrieval_documents_structured_v2.csv',
    index=False, encoding='utf-8-sig'
)

# Full CSV
pd.DataFrame([{
    'product_id':      d['product_id'],
    'product_name':    d['product_name'],
    'product_name_cn': d['product_name_cn'],
    'document':        to_single_line(d['document'])
} for d in full_docs]).to_csv(
    'retrieval_documents_full_v2.csv',
    index=False, encoding='utf-8-sig'
)

print('Saved → retrieval_documents_structured_v2.csv')
print('Saved → retrieval_documents_full_v2.csv')

from google.colab import files
files.download('retrieval_documents_structured_v2.csv')
files.download('retrieval_documents_full_v2.csv')

Saved → retrieval_documents_structured_v2.csv
Saved → retrieval_documents_full_v2.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>